# 01 — Verificación del entorno

**Objetivo:** confirmar que Python, las librerías y la estructura del proyecto están listas antes de tocar datos.

**Resultado que guarda:** `docs/entorno_versiones.txt` (versiones exactas, útil para reproducibilidad).

## 1. Hacer visible la carpeta `src/` desde el notebook
Los notebooks viven en `notebooks/`, pero el código reutilizable está en `src/`. Agregamos la raíz del repositorio a la ruta de Python para poder hacer `from src... import ...`.

In [1]:
import sys
from pathlib import Path

# Si el notebook se abre desde notebooks/, la raíz es la carpeta padre
RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print('Raíz del proyecto:', RAIZ)

Raíz del proyecto: C:\Users\Cindy\Proyecto ll\cobranza-ml


## 2. Versión de Python
Debe ser 3.12.x y estar corriendo desde `.venv` (el entorno virtual del proyecto).

In [2]:
import platform

print('Python:', platform.python_version())
print('Ejecutable:', sys.executable)

assert sys.version_info[:2] == (3, 12), 'Se esperaba Python 3.12: revisa el kernel seleccionado arriba a la derecha.'
assert '.venv' in sys.executable, 'El kernel no es el del entorno virtual .venv.'
print('✅ Python correcto y dentro de .venv')

Python: 3.12.10
Ejecutable: C:\Users\Cindy\Proyecto ll\cobranza-ml\.venv\Scripts\python.exe
✅ Python correcto y dentro de .venv


## 3. Librerías del proyecto
Importamos cada librería y mostramos su versión. Si alguna falla, el mensaje dice cuál instalar.

In [3]:
import importlib

librerias = {
    'pandas': 'pandas', 'numpy': 'numpy', 'pyarrow': 'pyarrow',
    'matplotlib': 'matplotlib', 'seaborn': 'seaborn',
    'sklearn': 'scikit-learn', 'lightgbm': 'lightgbm', 'shap': 'shap', 'joblib': 'joblib',
    'sqlalchemy': 'sqlalchemy', 'psycopg2': 'psycopg2-binary',
    'dotenv': 'python-dotenv', 'kagglehub': 'kagglehub', 'streamlit': 'streamlit',
}

versiones, faltantes = {}, []
for modulo, paquete in librerias.items():
    try:
        m = importlib.import_module(modulo)
        versiones[paquete] = getattr(m, '__version__', 'ok')
        print(f'✅ {paquete:18s} {versiones[paquete]}')
    except ImportError:
        faltantes.append(paquete)
        print(f'❌ {paquete:18s} NO instalado')

if faltantes:
    print('\nEjecuta en la terminal:  pip install', ' '.join(faltantes))

✅ pandas             3.1.0
✅ numpy              2.5.4
✅ pyarrow            26.0.0
✅ matplotlib         3.11.2


✅ seaborn            0.13.2
✅ scikit-learn       1.9.1
✅ lightgbm           4.7.0


C:\Users\Cindy\Proyecto ll\cobranza-ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ shap               0.54.1
✅ joblib             1.6.0


✅ sqlalchemy         2.1.6
✅ psycopg2-binary    2.9.13 (dt dec pq3 ext lo64)
✅ python-dotenv      ok


✅ kagglehub          1.0.3


✅ streamlit          1.67.0


## 4. Estructura del proyecto y configuración
Usamos `src/config.py` para crear las carpetas de datos y verificar que el archivo `.env` existe.

In [4]:
from src.config import RUTAS, crear_carpetas

crear_carpetas()
for nombre, ruta in RUTAS.items():
    print(f"{'✅' if ruta.exists() else '❌'} {nombre:15s} {ruta.relative_to(RUTAS['raiz'])}")

env = RUTAS['raiz'] / '.env'
print('\n.env encontrado ✅' if env.exists() else '\n⚠️ Falta .env: copia .env.example y renómbralo a .env')

✅ raiz            .
✅ data_raw        data\raw
✅ data_interim    data\interim
✅ data_processed  data\processed
✅ models          models
✅ sql             sql
✅ docs            docs
✅ figuras         docs\figuras

.env encontrado ✅


## 5. Guardar el reporte del entorno
Cada notebook termina guardando su resultado. Aquí guardamos las versiones para poder reproducir el proyecto en otro computador.

In [5]:
from datetime import datetime

salida = RUTAS['docs'] / 'entorno_versiones.txt'
lineas = [f'Fecha: {datetime.now():%Y-%m-%d %H:%M}', f'Python: {platform.python_version()}',
          f'Sistema: {platform.platform()}', ''] + [f'{k}=={v}' for k, v in versiones.items()]
salida.write_text('\n'.join(lineas), encoding='utf-8')
print('Reporte guardado en', salida.relative_to(RUTAS['raiz']))
print('✅ Entorno listo' if not faltantes else '⚠️ Revisa las librerías faltantes')

Reporte guardado en docs\entorno_versiones.txt
✅ Entorno listo
